# 1. Mount gdrive and import libs

In [10]:
from google.colab import drive
drive.mount('/content/gdrive')
import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Literal
import matplotlib.pyplot as plt

from skimage.filters import gaussian, threshold_otsu
from skimage.segmentation import *
import skimage.io as skio
from skimage.feature import graycomatrix, graycoprops
import skimage.morphology as skmor
import skimage.measure as skmea
from skimage.morphology import (
	disk,
	binary_opening,
	binary_closing,
	remove_small_objects,
	remove_small_holes
)

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, Subset

!pip install xgboost
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


# Unzip dataset zip file

In [11]:
#!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8"
#!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8"

# Dataset class definition:
Chatgpt5.4 is used for designing, debugging and adding boundary condition structure of this dataset class.

In [12]:
class Hep2Dataset(Dataset):
	def __init__(self, dataset_root, csv_path, transform=None):
		"""
		Initialize the dataset.

		Args:
			dataset_root (str or Path): Root folder of the image dataset.
			csv_path (str or Path): CSV file containing image file names and labels.
			transform (callable, optional): Transform to apply to each image.
		"""
		self.dataset_root = Path(dataset_root)
		self.csv_path = Path(csv_path)
		self.transform = transform

		# Read all samples once and store them as a unified list
		self.samples = self.read_label(self.csv_path)

	def read_label(self, csv_path):
		"""
		Read the CSV file and build a list of (image_path, label).

		Args:
			csv_path (Path): Path to the CSV label file.

		Returns:
			list: A list of tuples, where each tuple is (image_path, label).
		"""
		df = pd.read_csv(csv_path)

		samples = []
		for _, row in df.iterrows():
			img_name = row["file"]
			label = int(row["class"])  # Convert label to integer
			mask_path = self.dataset_root / f'{str(img_name).zfill(3)}_Mask.png'
			img_path = self.dataset_root / f'{str(img_name).zfill(3)}.png'
			samples.append((img_path, mask_path, label))

		return samples

	def __len__(self):
		"""
		Return the total number of samples in the dataset.
		"""
		return len(self.samples)

	def __getitem__(self, idx):
		"""
		Get one sample by index.

		Args:
			idx (int): Index of the sample.

		Returns:
			tuple: (image, mask, label)
		"""
		img_path, mask_path, label = self.samples[idx]
		img = skio.imread(img_path)
		mask = skio.imread(mask_path)

		if self.transform is not None:
			img = self.transform(img)

		return img, mask, label

# Shape feature extractor:
1.Chatgpt5.4 used for designing shape feature extraction class.

2.Here I used circularity, solidity and eccentricity beacuse we don't have metadata like what we have in DICOM format. So some features like perimeter and area, we cannnot get.

In [13]:
class ShapeFeatureExtractor:
	def __init__(self):
		return

	def _get_largest_region(self, mask):
		"""
		Get the largest connected component from a mask.

		Args:
			mask (ndarray): Input mask.

		Returns:
			regionprops object or None: Largest region if exists, else None.
		"""
		mask = mask > 0
		labeled_mask = skmea.label(mask)

		regions = skmea.regionprops(labeled_mask)
		if len(regions) == 0:
			return None

		largest_region = max(regions, key=lambda x: x.area)
		return largest_region

	def _get_solidity_from_region(self, region):
		"""
		Calculate solidity from a regionprops object.

		Args:
			region: regionprops object or None.
		Returns:
			float: Solidity value. Returns 0.0 if no valid region exists.
		"""
		if region is None:
			return 0.0

		return float(region.solidity)

	def _get_eccentricity_from_region(self, region):
		"""
		Calculate eccentricity from a regionprops object.

		Args:
			region: regionprops object or None.
		Returns:
			float: Eccentricity value. Returns 0.0 if no valid region exists.
		"""
		if region is None:
			return 0.0

		return float(region.eccentricity)

	def _get_circularity_from_region(self, region):
		"""
		Calculate circularity from a regionprops object.

		Formula:
			circularity = 4 * pi * area / perimeter^2

		Args:
			region: regionprops object or None.
		Returns:
			float: Circularity value. Returns 0.0 if no valid region exists
				   or perimeter is zero.
		"""
		if region is None:
			return 0.0

		area = region.area
		perimeter = region.perimeter

		if perimeter == 0:
			return 0.0

		circularity = 4.0 * np.pi * area / (perimeter ** 2)
		return float(circularity)


	def get_shape_features(self, mask):
		"""
		Extract all shape features from a single mask.

		Args:
			mask (ndarray): Input mask.

		Returns:
			dict: Dictionary containing shape features.
		"""
		largest_region = self._get_largest_region(mask)
		single_feature = {
			"circularity": self._get_circularity_from_region(largest_region),
			"solidity": self._get_solidity_from_region(largest_region),
			"eccentricity": self._get_eccentricity_from_region(largest_region),
		}
		return single_feature


# GLCM feature exatractor:

In [14]:
class GlcmFeatureExtractor:
	def __init__(self):
		"""
		Initialize the GLCM feature extractor.

		Args:
			image (ndarray): Input image.
			label (int): Corresponding label.
			mask (ndarray): Corresponding mask.
		"""

		# glcm configuration
		self.distance = 1
		self.angle = 0
		self.levels = 32
		self.symmetric = False
		self.normed = False

	def get_quantized_image(self, image):
		"""
		Get the gray-level image.

		Args:
			image (ndarray): Input image.
		Returns:
			ndarray: Gray-level image.
		"""
		quantized_gray_image = np.floor(image / (self.levels)).astype(np.uint8)
		return quantized_gray_image

	def get_masked_gray_image(self, image, mask):
		"""
		Get the gray-level image with the background masked out.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			ndarray: Gray-level image with background masked out.
		"""
		background_mask  = ~(mask > 0)
		gray_image = self.get_quantized_image(image)

		# only keep foreground region
		masked_image = gray_image.copy()
		masked_image[background_mask] = 0
		return masked_image

	def get_glcm_features(self, image, mask):
		glcm_features = {}

		# get masked gray image first
		masked_image = self.get_masked_gray_image(image, mask)

		# get GLCM matrix
		glcm = graycomatrix(
					masked_image.astype(np.uint8),
					distances=[self.distance],
					angles=[self.angle],
					levels=self.levels,
					symmetric=self.symmetric,
					normed=self.normed
					)
		# extract GLCM features
		glcm_contrast = graycoprops(glcm, prop='contrast')[0, 0]
		glcm_dissimilarity = graycoprops(glcm, prop='dissimilarity')[0, 0]
		glcm_homogeneity = graycoprops(glcm, prop='homogeneity')[0, 0]
		glcm_energy = graycoprops(glcm, prop='energy')[0, 0]
		glcm_correlation = graycoprops(glcm, prop='correlation')[0, 0]

		# store features in a dictionary
		glcm_features = {
			"contrast": glcm_contrast,
			"dissimilarity": glcm_dissimilarity,
			"homogeneity": glcm_homogeneity,
			"energy": glcm_energy,
			"correlation": glcm_correlation
		}

		return glcm_features


# Gray intensity feature extractor:

In [15]:
class ColorFeatureExtractor:
	def __init__(self):
		return

	def get_masked_gray_image(self, image, mask):
		"""
		Get the grayscale image with the background masked out.
		"""
		background_mask = ~(mask > 0)
		masked_image = image.copy()
		masked_image[background_mask] = 0
		return masked_image

	def get_gray_max_min(self, image, mask):
		"""
		Calculate the max and min pixel intensities in the masked image.
		"""
		masked_image = self.get_masked_gray_image(image, mask)
		max_gray = masked_image[mask > 0].max()
		min_gray = masked_image[mask > 0].min()
		return max_gray, min_gray

	def get_gray_avg(self, image, mask):
		"""
		Calculate the average pixel intensity in the masked image.
		"""
		masked_image = self.get_masked_gray_image(image, mask)
		avg_gray = masked_image[mask > 0].mean()
		return avg_gray

	def get_gray_std(self, image, mask):
		"""
		Calculate the standard deviation of pixel intensities in the masked image.
		"""
		masked_image = self.get_masked_gray_image(image, mask)
		std_gray = masked_image[mask > 0].std()
		return std_gray

	def get_gray_histogram(self, image, mask):
		"""
		Calculate the grayscale histogram of pixel intensities in the masked image.
		"""
		masked_image = self.get_masked_gray_image(image, mask)
		histogram, bin_edges = np.histogram(masked_image[mask > 0], bins=256, range=(0, 256))
		return histogram, bin_edges

	def get_gray_intensity_features(self, image, mask):
		"""
		Calculate a set of grayscale intensity features for a masked image.
		"""
		gray_features = {}
		gray_features["avg_intensity"] = self.get_gray_avg(image, mask)
		gray_features["std_intensity"] = self.get_gray_std(image, mask)
		gray_features["max_intensity"], gray_features["min_intensity"] = self.get_gray_max_min(image, mask)
		return gray_features


In [ ]:
train_dataset = Hep2Dataset(
	dataset_root="/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large",
	csv_path="/content/gdrive/MyDrive/BMET5933/WEEK_8/hep_img_large_gt.csv"
)



In [ ]:
# instantiate feature/ color extractors
shape_extractor = ShapeFeatureExtractor()
color_extractor = ColorFeatureExtractor()
glcm_extractor = GlcmFeatureExtractor()

feature_list = []
for i in range(len(train_dataset)):
	image, mask, label = train_dataset[i] # Unpack the tuple
	feature = {}
	# store features in a dictionary, and update the dictionary with different types of features
	feature.update(shape_extractor.get_shape_features(mask))
	feature.update(color_extractor.get_gray_intensity_features(image, mask))
	feature.update(glcm_extractor.get_glcm_features(image, mask))
	feature['label'] = label # Use the unpacked label
	feature_list.append(feature)

# convert into dataframe and extract x and y for training
feature_df = pd.DataFrame(feature_list)
x = feature_df.drop(columns=['label'])
y = feature_df['label']

# label converting
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# split the dataset into training and testing subsets, ratio is a experience-based choice
train_test_split_ratio = 0.2
x_train, x_test, y_train,y_test = train_test_split(
										x, 
										y_encoded, 
										test_size = 
										train_test_split_ratio, 
										random_state=42, 
										stratify=y_encoded
										)

,circularity,solidity,eccentricity,avg_intensity,std_intensity,max_intensity,min_intensity,contrast,dissimilarity,homogeneity,energy,correlation,label
0,0.894799,0.976247,0.480451,71.276399,11.667181,106,38,0.096955,0.095353,0.952484,0.623391,0.921331,1
1,0.887059,0.984387,0.697104,70.031344,6.900140,91,43,0.063661,0.063661,0.968169,0.686452,0.950519,1
2,0.901470,0.980896,0.616561,79.600212,13.430595,124,38,0.128838,0.121162,0.940186,0.588717,0.927626,1
3,0.867539,0.981395,0.753240,79.146327,13.509099,117,40,0.162273,0.156818,0.922136,0.575085,0.905755,1
4,0.876447,0.975348,0.599030,80.018132,9.907773,99,44,0.115352,0.083619,0.961364,0.715540,0.919793,1


# Classifier training

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate = 0.1
    objective='multi:softmax',
	eval_metric='mlogloss',
    random_state=42
    )

In [18]:
# These are for exporting the notebook as a PDF later if needed
!apt-get update
!sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc
!jupyter nbconvert --to pdf /content/gdrive/MyDrive/BMET5933/WEEK_8/Week_8-9_Li_Code_file_ipynb_draft_version.ipynb

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]      
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,862 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,142 kB]
Get:13